In [1]:
import tkinter as tk
from tkinter import messagebox
import math

# --- THẾ CỜ BAN ĐẦU ---
INITIAL_BOARD = [
    ['X', 'O', 'X'],
    ['O', 'X', ''],
    ['',  '',  'O']
]

AI = 'X'
PLAYER = 'O'

class SimpleTicTacToe:
    def __init__(self, root):
        self.root = root
        self.root.title("Tic Tac Toe Solver")

        # --- THANH CHỌN THUẬT TOÁN (TOP) ---
        top_frame = tk.Frame(root)
        top_frame.pack(pady=10)

        tk.Label(top_frame, text="Thuật toán:", font=('Arial', 11)).pack(side=tk.LEFT, padx=5)
        self.algo_var = tk.StringVar(value="Alpha-Beta")
        algo_menu = tk.OptionMenu(top_frame, self.algo_var, "Minimax", "Alpha-Beta", "Expectimax")
        algo_menu.config(font=('Arial', 10, 'bold'))
        algo_menu.pack(side=tk.LEFT, padx=5)

        # Nút bấm để kích hoạt AI giải bài toán
        solve_btn = tk.Button(top_frame, text="AI Giải Tiếp", font=('Arial', 10, 'bold'),
                              bg="#2ecc71", fg="white", command=self.ai_turn)
        solve_btn.pack(side=tk.LEFT, padx=10)

        # Nút Reset
        reset_btn = tk.Button(top_frame, text="Reset Thế Cờ", font=('Arial', 10),
                               bg="#e74c3c", fg="white", command=self.reset_game)
        reset_btn.pack(side=tk.LEFT, padx=5)

        # --- KHUNG BÀN CỜ CARO (CENTER) ---
        # Đã sửa lỗi pading thành công ở đây
        self.board_frame = tk.Frame(root, bg="#34495e")
        self.board_frame.pack(pady=10, padx=20)

        self.board = [row[:] for row in INITIAL_BOARD]
        self.buttons = [[None for _ in range(3)] for _ in range(3)]
        self.create_board_ui()
        self.update_ui()

    def create_board_ui(self):
        for r in range(3):
            for c in range(3):
                btn = tk.Button(self.board_frame, text="", font=('Arial', 24, 'bold'),
                                width=5, height=2, bg="#1caaa0", fg="white",
                                command=lambda row=r, col=c: self.player_move(row, col))
                btn.grid(row=r, column=c, padx=3, pady=3)
                self.buttons[r][c] = btn

    def update_ui(self):
        for r in range(3):
            for c in range(3):
                val = self.board[r][c]
                self.buttons[r][c].config(text=val)
                if val == 'X':
                    self.buttons[r][c].config(fg="#545454", state="disabled", disabledforeground="#545454")
                elif val == 'O':
                    self.buttons[r][c].config(fg="#efe7c8", state="disabled", disabledforeground="#efe7c8")
                else:
                    self.buttons[r][c].config(state="normal")

    def player_move(self, r, c):
        if self.board[r][c] == '':
            self.board[r][c] = PLAYER
            self.update_ui()
            self.check_game_over()

    def ai_turn(self):
        if self.check_game_over(): return

        algo = self.algo_var.get()
        best_move = self.get_best_move(algo)

        if best_move:
            r, c = best_move
            self.board[r][c] = AI
            self.update_ui()
            self.check_game_over()

    def reset_game(self):
        self.board = [row[:] for row in INITIAL_BOARD]
        self.update_ui()

    def check_game_over(self):
        winner = self.check_winner(self.board)
        if winner:
            if winner == 'Tie':
                messagebox.showinfo("Kết quả", "Hòa nhau!")
            else:
                messagebox.showinfo("Kết quả", f"Quân {winner} Thắng!")
            return True
        return False

    def check_winner(self, b):
        for i in range(3):
            if b[i][0] == b[i][1] == b[i][2] and b[i][0] != '': return b[i][0]
            if b[0][i] == b[1][i] == b[2][i] and b[0][i] != '': return b[0][i]
        if b[0][0] == b[1][1] == b[2][2] and b[0][0] != '': return b[0][0]
        if b[0][2] == b[1][1] == b[2][0] and b[0][2] != '': return b[0][2]
        if any('' in row for row in b): return None
        return 'Tie'

    def get_best_move(self, algo):
        best_score = -math.inf
        move = None

        for r in range(3):
            for c in range(3):
                if self.board[r][c] == '':
                    self.board[r][c] = AI

                    if algo == "Minimax":
                        score = self.minimax(self.board, 0, False)
                    elif algo == "Alpha-Beta":
                        score = self.alpha_beta(self.board, 0, False, -math.inf, math.inf)
                    elif algo == "Expectimax":
                        score = self.expectimax(self.board, 0, False)

                    self.board[r][c] = ''
                    if score > best_score:
                        best_score = score
                        move = (r, c)
        return move

    def minimax(self, b, depth, is_max):
        res = self.check_winner(b)
        if res == 'X': return 10 - depth
        if res == 'O': return depth - 10
        if res == 'Tie': return 0

        scores = []
        for r in range(3):
            for c in range(3):
                if b[r][c] == '':
                    b[r][c] = AI if is_max else PLAYER
                    scores.append(self.minimax(b, depth + 1, not is_max))
                    b[r][c] = ''
        return max(scores) if is_max else min(scores)

    def alpha_beta(self, b, depth, is_max, alpha, beta):
        res = self.check_winner(b)
        if res == 'X': return 10 - depth
        if res == 'O': return depth - 10
        if res == 'Tie': return 0

        if is_max:
            best = -math.inf
            for r in range(3):
                for c in range(3):
                    if b[r][c] == '':
                        b[r][c] = AI
                        best = max(best, self.alpha_beta(b, depth + 1, False, alpha, beta))
                        b[r][c] = ''
                        alpha = max(alpha, best)
                        if beta <= alpha: break
            return best
        else:
            best = math.inf
            for r in range(3):
                for c in range(3):
                    if b[r][c] == '':
                        b[r][c] = PLAYER
                        best = min(best, self.alpha_beta(b, depth + 1, True, alpha, beta))
                        b[r][c] = ''
                        beta = min(beta, best)
                        if beta <= alpha: break
            return best

    def expectimax(self, b, depth, is_max):
        res = self.check_winner(b)
        if res == 'X': return 10 - depth
        if res == 'O': return depth - 10
        if res == 'Tie': return 0

        empty_cells = [(r, c) for r in range(3) for c in range(3) if b[r][c] == '']

        if is_max:
            best = -math.inf
            for r, c in empty_cells:
                b[r][c] = AI
                best = max(best, self.expectimax(b, depth + 1, False))
                b[r][c] = ''
            return best
        else:
            total_score = 0
            for r, c in empty_cells:
                b[r][c] = PLAYER
                total_score += self.expectimax(b, depth + 1, True)
                b[r][c] = ''
            return total_score / len(empty_cells) if empty_cells else 0

if __name__ == "__main__":
    root = tk.Tk()
    root.eval('tk::PlaceWindow . center')
    app = SimpleTicTacToe(root)
    root.mainloop()